# 🧪 Water Quality Data Loading & Structure Diagnosis

**Author:** Gabriella Marín  
**Project:** Multi-Layer Water Quality Risk & Regulatory Analytics System
**Phase:** Phase 1 – Data Preparation
## Project Context
This notebook initiates Phase 1 of the project by loading the official water quality monitoring dataset
and performing a structural diagnosis.

The objective is to understand how the data is organized, identify key columns, detect data quality
issues, and assess parameter coverage before any cleaning or analytical decisions are made.
## Data Source

The dataset comes from official water quality monitoring records published through ''Datos Abiertos Colombia''.
It contains physicochemical measurements collected across different locations and dates between 2005 and 2024.

The data is expected to be heterogeneous, incomplete, and irregular, reflecting real-world monitoring conditions.


## Data Loading & Initial Inspection

In [8]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"D:\Documents\Portfolio\01-water-quality-normative")
DATA_FILE = PROJECT_DIR / "datos_calidad_del_agua_2005_2024.xlsx"

if not DATA_FILE.exists():
    raise FileNotFoundError(f"File not found: {DATA_FILE}")

df_raw = pd.read_excel(DATA_FILE, sheet_name="BASE DE DATOS", header=5)

print("Dataset shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes)

Dataset shape: (134261, 17)


,Unnamed: 0,NOMBRE DEL PUNTO DE MONITOREO,LATITUD,LONGITUD,ELEVACIÓN (m.s.n.m.),CORRIENTE,ZONA HIDROGRÁFICA - ZH,SZH - Código (#Área#Zona##Subzona),Nombre Subzona Hidrográfica,DEPARTAMENTO,MUNICIPIO,FECHA,PROPIEDAD OBSERVADA,*RESULTADO,UNIDAD DEL RESULTADO,PROYECTO,CODIGO__MUESTRA
0,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,AGUADAS,2007-03-09,ALUMINIO POTENCIALMENTE BIODISPONIBLE,5339,mg Al/kg,Ideam,14615
1,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,AGUADAS,2013-11-09,ALUMINIO POTENCIALMENTE BIODISPONIBLE,12560,mg Al/Kg,Ideam,22820
2,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,AGUADAS,2016-11-07,ALUMINIO POTENCIALMENTE BIODISPONIBLE,4408,mg Al/Kg,Ideam,25379
3,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,AGUADAS,2013-11-09,ALUMINIO TOTAL EN AGUA,12,mg Al/L,Ideam,22820
4,NaN,ARMA_CAL_AGUADAS_DESEMBOCADURA RIO ARMA,5.7375,-75.5975,612,ARMA,Cauca,2618,Río Arma,CALDAS,AGUADAS,2016-11-07,ALUMINIO TOTAL EN AGUA,7.31,mg Al/L,Ideam,25379


Unnamed: 0                                   float64
NOMBRE DEL PUNTO DE MONITOREO                    str
LATITUD                                      float64
LONGITUD                                     float64
ELEVACIÓN (m.s.n.m.)                           int64
CORRIENTE                                        str
ZONA HIDROGRÁFICA - ZH                           str
SZH - Código (#Área#Zona##Subzona)             int64
Nombre Subzona Hidrográfica                      str
DEPARTAMENTO                                     str
MUNICIPIO                                        str
FECHA                                 datetime64[us]
PROPIEDAD OBSERVADA                              str
*RESULTADO                                    object
UNIDAD DEL RESULTADO                             str
PROYECTO                                         str
CODIGO__MUESTRA                                int64
dtype: object

## Structural Diagnosis

In [9]:
# Column names
cols = list(df_raw.columns)
cols_lower = [c.lower() for c in cols]

print("Columns:")
display(cols)

# Dataset format diagnosis
looks_long = (any("propiedad" in c for c in cols_lower)
            and any("unidad" in c for c in cols_lower)
            and any(("resultado" in c) or ("result" in c) for c in cols_lower))

print("Dataset format:", "LONG" if looks_long else "WIDE or MIXED")

# Key column detection
def find_col_contains(keywords):
    for c in cols:
        cl = c.lower()
        if all(k in cl for k in keywords):
            return c
    return None

date_col = find_col_contains(["fecha"])
param_col = find_col_contains(["propiedad"])
value_col = find_col_contains(["resultado"])
unit_col  = find_col_contains(["unidad"])

print("Detected key columns:")
print("Date column:", date_col)
print("Parameter column:", param_col)
print("Value column:", value_col)
print("Unit column:", unit_col)

Columns:


['Unnamed: 0',
 'NOMBRE DEL PUNTO DE MONITOREO',
 'LATITUD',
 'LONGITUD',
 'ELEVACIÓN (m.s.n.m.)',
 'CORRIENTE',
 'ZONA HIDROGRÁFICA - ZH',
 'SZH - Código (#Área#Zona##Subzona)',
 'Nombre Subzona Hidrográfica',
 'DEPARTAMENTO',
 'MUNICIPIO',
 'FECHA',
 'PROPIEDAD OBSERVADA ',
 '*RESULTADO',
 'UNIDAD DEL RESULTADO',
 'PROYECTO',
 'CODIGO__MUESTRA']

Dataset format: LONG
Detected key columns:
Date column: FECHA
Parameter column: PROPIEDAD OBSERVADA 
Value column: *RESULTADO
Unit column: UNIDAD DEL RESULTADO


## Data Quality Diagnostics

In [10]:
# Missing values overview
na_pct = (df_raw.isna().mean().sort_values(ascending=False).mul(100).round(1))
display(na_pct.head(15))

# Value pattern diagnostics
s = df_raw[value_col].dropna().astype(str).str.strip()

patterns = {"censored_less_than": s.str.startswith("<").mean(),
            "text_values": s.str.contains(r"[A-Za-z]", regex=True).mean(),
            "missing_tokens": s.str.contains(r"ND|N/A|SIN DATO", case=False, regex=True).mean(),}

print("Value patterns:")
for k, v in patterns.items():
    print(f"{k}: {v:.2%}")

## Parameter Coverage Overview
param_counts = df_raw[param_col].value_counts(dropna=True)
display(param_counts.head(20))

Unnamed: 0                            100.0
NOMBRE DEL PUNTO DE MONITOREO           0.0
LATITUD                                 0.0
LONGITUD                                0.0
ELEVACIÓN (m.s.n.m.)                    0.0
CORRIENTE                               0.0
ZONA HIDROGRÁFICA - ZH                  0.0
SZH - Código (#Área#Zona##Subzona)      0.0
Nombre Subzona Hidrográfica             0.0
DEPARTAMENTO                            0.0
MUNICIPIO                               0.0
FECHA                                   0.0
PROPIEDAD OBSERVADA                     0.0
*RESULTADO                              0.0
UNIDAD DEL RESULTADO                    0.0
dtype: float64

Value patterns:
censored_less_than: 28.65%
text_values: 0.00%
missing_tokens: 0.00%


PROPIEDAD OBSERVADA 
CONDUCTIVIDAD ELECTRICA                6771
pH                                     6764
TEMPERATURA                            6738
DEMANDA QUIMICA DE OXIGENO (DQO)       6694
SOLIDOS SUSPENDIDOS TOTALES            6660
OXIGENO DISUELTO (OD)                  6559
TURBIDEZ                               6549
NITRITO                                5129
FOSFORO REACTIVO DISUELTO              5113
NITRATO                                5101
NITROGENO AMONIACAL                    4997
FOSFORO TOTAL                          4966
SULFATO                                4199
NITROGENO KJELDAHL TOTAL               3472
CADMIO POTENCIALMENTE BIODISPONIBLE    2316
ZINC TOTAL EN AGUA                     2087
NIQUEL TOTAL EN AGUA                   2086
COBRE TOTAL EN AGUA                    2085
CROMO TOTAL EN AGUA                    2082
COBRE POTENCIALMENTE BIODISPONIBLE     2060
Name: count, dtype: int64

## Summary

The dataset was successfully loaded and diagnosed.
It follows a long-format structure with consistent key columns for date, parameter, value, and unit.

No structural missing values were detected; however, a significant proportion of measurements are reported
as censored values below detection limits, which must be handled during data standardization.

Parameter coverage is strong for physicochemical and nutrient-related variables, while microbiological
parameters are insufficient for analytical modeling.

These findings define the scope and transformations applied in the next phase.